# Task C exploration: raw detector bits

Goal of this notebook: understand the data Task C will train on before writing any model code.

Task C uses only the Google ML table, filtered to `distance = 3 AND shot_index < 12_500`, with the supplied `data_split`.
Inputs are the 200 unpacked detector bits; the target is `actual_observable_flip`.

## 1. Load the Google ML table from the object store

Part II may read only the two `ml/` Parquet tables, never Gold, Silver, or Bronze.

In [ ]:
import io

import numpy as np
import pandas as pd

from quantum_lake_student.config import Settings
from quantum_lake_student.connections import read_lake_object
from quantum_lake_student.ml import (
    GOOGLE_META_PREDICTION_COLUMNS,
    google_data_split,
    unpack_little_endian_bits,
    weighted_logical_error_rate,
)

settings = Settings.from_environment()
raw = read_lake_object(settings, "ml/ml_google_decoder_example.parquet")
google = pd.read_parquet(io.BytesIO(raw))
google.shape

In [ ]:
google.dtypes

In [ ]:
google.head()

## 2. checking experiments

Expect five experiments: four distance-3 processor locations and one distance-5 location, each with 50,000 shots.

In [ ]:
google.groupby(["experiment_id", "distance", "rounds", "center_row", "center_col"]).agg(
    shots=("shot_index", "size"),
    min_shot=("shot_index", "min"),
    max_shot=("shot_index", "max"),
    detector_count=("detector_count", "unique"),
)

## 3. Apply the Task C filter

Expected: 4 experiments x 12,500 shots = 50,000 rows.

In [ ]:
subset = google[(google["distance"] == 3) & (google["shot_index"] < 12_500)].reset_index(drop=True)
print(len(subset), "rows")
subset.groupby("experiment_id").size()

## 4. Check the supplied split

Expected per experiment: 5,000 train, 1,250 validation, 6,250 test, so 20,000 / 5,000 / 25,000 overall.
The stored data_split should also agree with the supplied google_data_split helper.

In [ ]:
print(subset["data_split"].value_counts())

recomputed = subset["shot_index"].map(google_data_split)
print("split matches helper:", (recomputed == subset["data_split"]).all())

pd.crosstab(subset["experiment_id"], subset["data_split"])

## 5. Unpack the detector bits

Each distance-3 row should have `detector_count = 200`, so the packed value is `ceil(200 / 8) = 25` bytes.
200 is a multiple of 8, so there are no padding bits for distance 3.

Check: the number of set bits after unpacking must equal `detector_event_count`.

In [ ]:
print("detector_count values:", subset["detector_count"].unique())
print("packed byte lengths:", subset["detector_bits"].map(len).unique())

X = np.array(
    [unpack_little_endian_bits(packed, 200) for packed in subset["detector_bits"]],
    dtype=np.uint8,
)
print("feature matrix:", X.shape)
print("event counts match:", (X.sum(axis=1) == subset["detector_event_count"].to_numpy()).all())

## 6. Label balance

How often does the logical observable actually flip? for threshold choice and for reading balanced accuracy

In [ ]:
subset.groupby("data_split")["actual_observable_flip"].agg(["mean", "sum", "size"])

## 7. How sparse are the inputs?

Event density per shot, and how often each of the 200 detectors fires.

In [ ]:
print((subset["detector_event_count"] / subset["detector_count"]).describe())

firing_rate = pd.Series(X.mean(axis=0), name="firing_rate")
firing_rate.describe()

In [ ]:
pd.DataFrame(
    firing_rate.to_numpy().reshape(25, 8),
    index=[f"round_{r}" for r in range(25)],
    columns=[f"det_{d}" for d in range(8)],
).round(3)

## 8. Reference point: supplied decoders on the same test shots

tells what logical-error rate a good decoder reaches on exactly the rows 

In [ ]:
test = subset[subset["data_split"] == "test"]
ones = np.ones(len(test))

rows = {
    column: weighted_logical_error_rate(test["actual_observable_flip"], test[column], ones)
    for column in GOOGLE_META_PREDICTION_COLUMNS
}
majority = bool(subset.loc[subset["data_split"] == "train", "actual_observable_flip"].mean() >= 0.5)
rows["majority_baseline"] = weighted_logical_error_rate(
    test["actual_observable_flip"], [majority] * len(test), ones
)
pd.Series(rows, name="test_logical_error_rate").sort_values()

## 9. notes
- Subset row count and split counts:
    - Test: odd indices, 6,250 per experiment.
    - -Validation: even indices ending in 8, 1,250 per experiment.
    - Train: the remaining even indices, 5,000 per experiment.
- Event counts match unpacked bits?:
    - 200 bits per shot, 25 bytes packed. Unpacked set-bit count equals detector_event_count for all 50,000 rows
- Label rate (train / validation / test):
    - 0.496 / 0.494 / 0.500.  balanced, roughly 50% flips in every split
- Does the 25 x 8 firing-rate grid show round structure?:
    - Yes. round_0 and round_24 fire less often than the middle rounds, and some detector columns fire less in every round (about 0.11 vs about 0.17), likely edge vs central stabilizers. Supports the 25 rounds x 8 detectors ordering, which a flat MLP input does not see.
- Best supplied decoder LER on the test shots:
    - tensor_network_contraction = 0.40000 (belief_matching 0.40020, correlated_matching 0.42116, pymatching 0.43228)
- Majority baseline LER on the test shots: 0.50008
- Bit order check passed (0b101 -> 1,0,1,0,...); no padding bits for distance 3 (200 = 25 x 8)
- Takeaway: supplied decoders reach ~0.40 LER vs 0.50 baseline; the MLP is expected to land between these


# running the trained model

In [ ]:
from quantum_lake_student.task_c import build_features, train_mlp, evaluate

f = build_features(google)        
trained = train_mlp(f, seed=7)   
result = evaluate(f, trained)

print("threshold:", result.threshold)
pd.DataFrame(result.metrics)

# Interpreting results

- **Threshold:** 0.6, chosen on the 5,000 validation shots only.
- **Test LER:** MLP 0.4993 vs majority baseline 0.5001, so the MLP performs at chance level. The supplied decoders reach about 0.40 on the same test shots, so the bits do contain a signal; the flat MLP cannot extract it.
- **Validation vs test:** validation looks slightly better (0.4886), but the threshold was tuned on those rows, so that number is optimistic. The official result is the test score.
- **Balanced accuracy:** 0.501 for the MLP and exactly 0.500 for the baseline, which always predicts "no flip". Labels are about 50/50, so this agrees with the LER.
- **Brier score:** the MLP (0.2556) is slightly worse than the baseline (0.2500). Its probabilities are overconfident: it leans away from 0.5 but is wrong as often as it is right.
- **Conclusion:** the MLP memorises the training shots but does not generalise. The 200 bits are given as a flat vector without the 25-round × 8-detector structure, and 20,000 examples are too few to learn that structure from scratch.
